# Graduation Project: EEG-to-Image Reconstruction
### Visual Stimulus Reconstruction from Brain Signals using Stable Diffusion 2.1 & ControlNet

---

### Project Overview
The goal of this project is to decode human visual perception directly from non-invasive brain activity (EEG). When a subject views an image, 128-channel EEG signals are recorded. Our pipeline processes these brain signals and guides a latent diffusion model to reconstruct the visual image the subject was looking at.

### System Architecture
1. **Semantic Decoder:** A 4-layer LSTM network (`EEGFeatNet`) + k-NN classifier to predict the object category (ImageNet class) from EEG, which is then mapped into text prompt embeddings via CLIP.
2. **Structural ControlNet:**
   - `SubjectLayers`: Calibrates individual skull and electrode variations across subjects into a shared space.
   - `EEGConditionEmbedding`: 1D convolutional layers converting temporal EEG signals into 2D spatial feature maps (64x64) for ControlNet.
3. **Generative Backbone:** **Stable Diffusion 2.1** with frozen UNet weights to preserve generation quality and save GPU memory, training only the ControlNet branches.


## 1. Environment & GPU Verification
Verify that PyTorch, Diffusers, Accelerate, and CUDA GPU acceleration are available.


In [ ]:
import torch
import accelerate
import diffusers

from diffusers import (
    AutoencoderKL,
    UNet2DConditionModel,
    DDPMScheduler,
    ControlNetModel,
)

print("PyTorch Version:   ", torch.__version__)
print("Accelerate Version:", accelerate.__version__)
print("Diffusers Version: ", diffusers.__version__)
print("CUDA Available:    ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:        ", torch.cuda.get_device_name(0))

print("Environment verified successfully.")


### Library Imports & Compatibility Setup
Automatically install missing dependencies and import deep learning, image, and signal processing packages.


In [ ]:
import os, sys, glob, time, random, pickle, urllib.request, warnings, copy
warnings.filterwarnings("ignore")

# Automatically install missing packages in Kaggle environment
required = {
    "diffusers": "diffusers",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "tqdm": "tqdm",
    "sklearn": "scikit-learn",
}
import importlib.util, subprocess
for module_name, package_name in required.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm
from torchvision import transforms
from sklearn.neighbors import KNeighborsClassifier
import matplotlib.pyplot as plt

try:
    from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, ControlNetModel
except AttributeError:
    for module_name in [m for m in list(sys.modules) if m.startswith("diffusers")]:
        del sys.modules[module_name]
    sys.modules["torch._dynamo"] = None
    from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, ControlNetModel

from transformers import CLIPTextModel, CLIPTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected. On Kaggle: Settings -> Accelerator -> GPU and rerun.")

DEVICE = torch.device("cuda")
print("Device ready:", torch.cuda.get_device_name(0))


## 2. Paths & Hyperparameters
Configure dataset directories, output checkpoints, and training settings:
- `RES = 512`: Target image resolution (512x512).
- `BATCH_SIZE = 4` with `GRAD_ACCUM = 2`: Effective batch size of 8 while saving GPU VRAM.
- `PROP_EMPTY = 0.5`: 50% empty-text probability to train Classifier-Free Guidance (CFG).
- `TIME_BUDGET_H = 8.0`: Maximum training time limit to safely save checkpoints before Kaggle session timeout.


In [ ]:
DATA_ROOT = "/kaggle/input/datasets/evolutionaw/eegcvpr40/data"
IMAGE_DIR = "/kaggle/input/competitions/imagenet-object-localization-challenge/ILSVRC/Data/CLS-LOC/train"

WORK_DIR = "/kaggle/working"
ASSET_DIR = os.path.join(WORK_DIR, "gwit_assets")
LEGACY_CHECKPOINT_PATH = os.path.join(WORK_DIR, "exact-paper-modelV00.pt")
OUT_PATH = os.path.join(WORK_DIR, "exact-paper-modelV00_6subjects.pt")
BEST_PATH = os.path.join(WORK_DIR, "best_gwit_model_6subjects.pt")

SD_IDS = [
    "stabilityai/stable-diffusion-2-1-base",
    "sd2-community/stable-diffusion-2-1-base",
]
GH = "https://raw.githubusercontent.com/LuigiSigillo/GWIT/main/src/"

# Training Hyperparameters
RES = 512
LR = 1e-5
WEIGHT_DECAY = 1e-2
MAX_GRAD_NORM = 1.0
EPOCHS = 20
PROP_EMPTY = 0.5
SEED = 42
BATCH_SIZE = 4
GRAD_ACCUM = 2
TIME_BUDGET_H = 8.0
DECODER_MIN_ACC = 0.85
DECODER_EPOCHS = 150
N_STORE_TEST = 400
SAVE_EVERY_STEPS = 1000
LATENT_BATCH_SIZE = 8
FINETUNE_DECODER_IF_NEEDED = True

os.makedirs(ASSET_DIR, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def fetch_asset(relative_path):
    """Download pretrained assets from official repo if not present locally."""
    local_path = os.path.join(ASSET_DIR, os.path.basename(relative_path))
    if not os.path.exists(local_path):
        url = GH + relative_path
        print("Downloading asset:", url)
        urllib.request.urlretrieve(url, local_path)
    return local_path

if not os.path.isdir(DATA_ROOT):
    raise FileNotFoundError(f"DATA_ROOT not found: {DATA_ROOT}")
if not os.path.isdir(IMAGE_DIR):
    raise FileNotFoundError(f"IMAGE_DIR not found: {IMAGE_DIR}")

print("Paths and hyperparameters configured successfully.")


## 3. EEG Data Loading & Image-Level Splitting
- **Signal Filtering:** Load 128-channel EEG recordings and extract the 440-time-step window (samples 20 to 460), corresponding to Visual Evoked Potentials (VEP).
- **Image Linking:** Map each EEG trial to its corresponding ImageNet visual stimulus.
- **Zero-Leakage Splitting:** Split the data (Train 70%, Val 15%, Test 15%) at the **image level** so training images never appear in validation or testing.


In [ ]:
CKPT_FEAT = fetch_asset(
    "EEGStyleGAN-ADA/EEG2Feat/Triplet_LSTM/CVPR40/EXPERIMENT_29/bestckpt/eegfeat_all_0.9665178571428571.pth"
)
KNN_PKL = fetch_asset("gwit/dataset_EEG/knn_model.pkl")
name_map_path = fetch_asset("gwit/dataset_EEG/name_map_ID.py")

name_space = {}
with open(name_map_path, "r", encoding="utf-8") as f:
    exec(compile(f.read(), name_map_path, "exec"), name_space)

id_to_caption = name_space["id_to_caption"]
folder_label_map = name_space["folder_label_map"]
caption_to_id = {caption: int(class_id) for class_id, caption in id_to_caption.items()}
wordnet_to_id = {
    wn: caption_to_id[label]
    for wn, label in folder_label_map.items()
    if label in caption_to_id
}

# Search for data and split files
pth_files = glob.glob(os.path.join(DATA_ROOT, "**", "*.pth"), recursive=True)
split_candidates = [p for p in pth_files if "split" in os.path.basename(p).lower()]
data_candidates = [p for p in pth_files if "split" not in os.path.basename(p).lower()]
if not data_candidates:
    raise FileNotFoundError(f"No .pth files found in {DATA_ROOT}")

eeg_file = max(data_candidates, key=os.path.getsize)
split_file = None
if split_candidates:
    split_file = sorted(
        split_candidates,
        key=lambda p: ("all" not in os.path.basename(p).lower(), os.path.basename(p))
    )[0]

print("EEG File:  ", os.path.basename(eeg_file))
print("Split File:", os.path.basename(split_file) if split_file else "Dynamic Image-Disjoint Split")

# Load and filter EEG trials
data_obj = torch.load(eeg_file, map_location="cpu", weights_only=False)
image_names = data_obj["images"]
dataset_rows = data_obj["dataset"]

X_list, ORIGINAL_ROW, IMAGE_IDS, SUBJECT_IDS = [], [], [], []
for original_index, item in enumerate(tqdm(dataset_rows, desc="Reading EEG trials")):
    eeg = torch.as_tensor(item["eeg"]).float()
    if eeg.ndim != 2:
        continue
    if eeg.shape[0] != 128 and eeg.shape[1] == 128:
        eeg = eeg.t()
    if eeg.shape[0] != 128 or not (450 <= eeg.shape[1] <= 600):
        continue
    X_list.append(eeg[:, 20:460].contiguous())
    ORIGINAL_ROW.append(original_index)
    IMAGE_IDS.append(int(item["image"]))
    SUBJECT_IDS.append(int(item["subject"]))

del data_obj, dataset_rows
if not X_list:
    raise RuntimeError("No valid EEG trials loaded.")

X = torch.stack(X_list).contiguous()           # Shape: [N, 128 channels, 440 time steps]
ORIGINAL_ROW = np.asarray(ORIGINAL_ROW, dtype=np.int64)
IMAGE_IDS = np.asarray(IMAGE_IDS, dtype=np.int64)
SUBJECT_ORIGINAL = np.asarray(SUBJECT_IDS, dtype=np.int64)

# Match EEG trials to image filepaths
def find_image_path(image_name):
    folder = image_name.split("_")[0]
    candidate = os.path.join(IMAGE_DIR, folder, image_name + ".JPEG")
    if os.path.isfile(candidate):
        return candidate
    matches = glob.glob(os.path.join(IMAGE_DIR, "**", image_name + ".*"), recursive=True)
    return matches[0] if matches else None

unique_image_ids = sorted(set(IMAGE_IDS.tolist()))
IMAGE_PATHS = {image_id: find_image_path(image_names[image_id]) for image_id in unique_image_ids}
valid_mask = np.asarray([IMAGE_PATHS[image_id] is not None for image_id in IMAGE_IDS], dtype=bool)

X = X[torch.from_numpy(valid_mask)]
ORIGINAL_ROW = ORIGINAL_ROW[valid_mask]
IMAGE_IDS = IMAGE_IDS[valid_mask]
SUBJECT_ORIGINAL = SUBJECT_ORIGINAL[valid_mask]
IMAGE_WORDS = [image_names[i].split("_")[0] for i in IMAGE_IDS]
N = len(IMAGE_IDS)

# Remap subject identifiers to contiguous 0..N_SUBJECTS-1 indices
SUBJECT_VALUES = sorted(np.unique(SUBJECT_ORIGINAL).tolist())
SUBJECT_LOOKUP = {subject_id: idx for idx, subject_id in enumerate(SUBJECT_VALUES)}
SUBJECTS = np.asarray([SUBJECT_LOOKUP[s] for s in SUBJECT_ORIGINAL], dtype=np.int64)
N_SUBJECTS = len(SUBJECT_VALUES)

if N < 20:
    raise RuntimeError(f"Too few valid samples ({N}). Verify paths and data.")

rows_by_image = {}
for row_index, image_id in enumerate(IMAGE_IDS):
    rows_by_image.setdefault(int(image_id), []).append(row_index)

rng = np.random.RandomState(SEED)
if split_file is not None:
    split_obj = torch.load(split_file, map_location="cpu", weights_only=False)
    official_split = split_obj["splits"][0]
    original_to_current = {int(original_idx): row_idx for row_idx, original_idx in enumerate(ORIGINAL_ROW)}

    def map_split(key):
        return np.asarray([
            original_to_current[int(idx)]
            for idx in official_split[key]
            if int(idx) in original_to_current
        ], dtype=np.int64)

    train_rows = map_split("train")
    val_rows = map_split("val")
    test_rows = map_split("test")
else:
    all_image_ids = np.asarray(sorted(rows_by_image.keys()), dtype=np.int64)
    rng.shuffle(all_image_ids)
    n_images = len(all_image_ids)
    n_train_images = int(0.70 * n_images)
    n_val_images = int(0.15 * n_images)
    train_image_set = set(all_image_ids[:n_train_images].tolist())
    val_image_set = set(all_image_ids[n_train_images:n_train_images+n_val_images].tolist())
    test_image_set = set(all_image_ids[n_train_images+n_val_images:].tolist())

    def rows_for_image_set(image_set):
        return np.asarray([
            row for image_id in image_set for row in rows_by_image[int(image_id)]
        ], dtype=np.int64)

    train_rows = rows_for_image_set(train_image_set)
    val_rows = rows_for_image_set(val_image_set)
    test_rows = rows_for_image_set(test_image_set)

def image_overlap(rows_a, rows_b):
    return len(set(IMAGE_IDS[rows_a].tolist()) & set(IMAGE_IDS[rows_b].tolist()))

print(f"Dataset split: Total={N} | Train={len(train_rows)} | Val={len(val_rows)} | Test={len(test_rows)}")
print(f"Subjects: {N_SUBJECTS} | IDs: {SUBJECT_VALUES}")
print("Image overlap train/val (Expected 0):", image_overlap(train_rows, val_rows))
print("Image overlap train/test (Expected 0):", image_overlap(train_rows, test_rows))


## 4. EEG Semantic Decoder & Classification
- `EEGFeatNet` (4-layer LSTM): Maps EEG signals into 128-dimensional semantic embeddings.
- `k-NN Classifier`: Classifies the embedding into one of 40 visual categories.
- **Adaptive Fine-Tuning:** If validation accuracy is under 85%, fine-tunes the LSTM on ground-truth labels to improve classification.


In [ ]:
def normalize_eeg(batch):
    # batch: [Batch, channels=128, time=440]
    scale = batch.amax(dim=(1, 2), keepdim=True).clamp_min(1e-6) / 2.0
    return (batch - scale) / scale

class EEGFeatNet(nn.Module):
    def __init__(self, in_channels=128, n_features=128, projection_dim=128, num_layers=4):
        super().__init__()
        self.hidden_size = n_features
        self.num_layers = num_layers
        self.encoder = nn.LSTM(
            input_size=in_channels,
            hidden_size=n_features,
            num_layers=num_layers,
            batch_first=True,
        )
        self.fc = nn.Linear(n_features, projection_dim, bias=False)

    def forward(self, x):
        # x: [Batch, time=440, channels=128]
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size, device=x.device)
        c0 = torch.zeros_like(h0)
        _, (hidden, _) = self.encoder(x, (h0, c0))
        return F.normalize(self.fc(hidden[-1]), dim=-1)

feature_checkpoint = torch.load(CKPT_FEAT, map_location="cpu", weights_only=False)
featnet = EEGFeatNet().to(DEVICE)
state_dict = {
    key.replace("module.", "", 1) if key.startswith("module.") else key: value
    for key, value in feature_checkpoint["model_state_dict"].items()
}
featnet.load_state_dict(state_dict)
featnet.eval()
del feature_checkpoint

with open(KNN_PKL, "rb") as f:
    knn = pickle.load(f)

@torch.no_grad()
def extract_decoder_features(rows, batch_size=128):
    feature_parts = []
    for start in range(0, len(rows), batch_size):
        row_batch = np.asarray(rows[start:start + batch_size], dtype=np.int64)
        eeg_batch = normalize_eeg(X[torch.as_tensor(row_batch)].to(DEVICE))
        features = featnet(eeg_batch.transpose(1, 2))
        feature_parts.append(features.cpu().numpy())
    if not feature_parts:
        return np.empty((0, 128), dtype=np.float32)
    return np.concatenate(feature_parts, axis=0)

FEATURES = extract_decoder_features(np.arange(N))
CAPTIONS = np.asarray(knn.predict(FEATURES), dtype=np.int64)

def decoder_accuracy(rows):
    if len(rows) == 0:
        return float("nan")
    correct, total = 0, 0
    for row in rows:
        predicted_caption = id_to_caption.get(int(CAPTIONS[row]))
        true_caption = folder_label_map.get(IMAGE_WORDS[row])
        if predicted_caption is not None and true_caption is not None:
            correct += int(predicted_caption == true_caption)
            total += 1
    return correct / total if total else 0.0

initial_decoder_acc = decoder_accuracy(val_rows)
print(f"Pretrained Decoder Validation Accuracy: {initial_decoder_acc * 100:.2f}%")

word_labels = np.asarray([wordnet_to_id.get(word, -1) for word in IMAGE_WORDS], dtype=np.int64)
valid_label_rows = np.flatnonzero((word_labels >= 0) & (word_labels < 40))

if FINETUNE_DECODER_IF_NEEDED and initial_decoder_acc < DECODER_MIN_ACC:
    valid_train_rows = np.intersect1d(train_rows, valid_label_rows, assume_unique=False)
    valid_val_rows = np.intersect1d(val_rows, valid_label_rows, assume_unique=False)
    if len(valid_train_rows) >= 128 and len(valid_val_rows) > 0:
        print("Fine-tuning EEG decoder with Cosine Softmax Loss...")
        featnet = EEGFeatNet().to(DEVICE)
        classification_head = nn.Linear(128, 40, bias=False).to(DEVICE)
        decoder_optimizer = torch.optim.AdamW(
            list(featnet.parameters()) + list(classification_head.parameters()),
            lr=1e-3,
            weight_decay=1e-2,
        )
        best_decoder_acc = -1.0
        best_decoder_state = None

        for decoder_epoch in range(DECODER_EPOCHS):
            featnet.train()
            shuffled = np.random.permutation(valid_train_rows)
            for start in range(0, len(shuffled) - 127, 128):
                batch_rows = shuffled[start:start + 128]
                eeg_batch = normalize_eeg(X[torch.as_tensor(batch_rows)].to(DEVICE))
                features = featnet(eeg_batch.transpose(1, 2))
                logits = 16.0 * F.normalize(features, dim=-1) @ F.normalize(classification_head.weight, dim=-1).T
                targets = torch.as_tensor(word_labels[batch_rows], dtype=torch.long, device=DEVICE)
                loss = F.cross_entropy(logits, targets)
                decoder_optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(featnet.parameters(), 1.0)
                decoder_optimizer.step()

            if decoder_epoch % 5 == 4 or decoder_epoch == DECODER_EPOCHS - 1:
                featnet.eval()
                current_features = extract_decoder_features(np.arange(N))
                knn_candidate = KNeighborsClassifier(n_neighbors=3)
                knn_candidate.fit(current_features[valid_train_rows], word_labels[valid_train_rows])
                val_predictions = knn_candidate.predict(current_features[valid_val_rows])
                current_acc = float(np.mean(val_predictions == word_labels[valid_val_rows]))
                if current_acc > best_decoder_acc:
                    best_decoder_acc = current_acc
                    best_decoder_state = copy.deepcopy(featnet.state_dict())

        if best_decoder_state is not None:
            featnet.load_state_dict(best_decoder_state)
            featnet.eval()
            FEATURES = extract_decoder_features(np.arange(N))
            knn = KNeighborsClassifier(n_neighbors=3)
            knn.fit(FEATURES[valid_train_rows], word_labels[valid_train_rows])
            CAPTIONS = np.asarray(knn.predict(FEATURES), dtype=np.int64)
            print(f"Best Fine-Tuned Accuracy: {best_decoder_acc * 100:.2f}%")

print(f"Final Decoder Validation Accuracy: {decoder_accuracy(val_rows) * 100:.2f}%")


## 5. Stable Diffusion & VAE Latent Precomputation
- **CLIP Text Embeddings:** Compute embeddings for the 40 category prompts and empty prompt (for CFG), then remove the text encoder from GPU to free VRAM.
- **Precomputed VAE Latents:** Encode images once into compressed latent representations (4x64x64) and store them in **CPU RAM** to avoid GPU Out-of-Memory (OOM) errors during training.


In [ ]:
SD_ID = None
for candidate_id in SD_IDS:
    try:
        UNet2DConditionModel.load_config(candidate_id, subfolder="unet")
        SD_ID = candidate_id
        break
    except Exception as exc:
        print(f"Could not load {candidate_id}: {exc}")

if SD_ID is None:
    raise RuntimeError("Could not access Stable Diffusion 2.1 weights.")

# Precompute CLIP text embeddings and delete encoder to free VRAM
tokenizer = CLIPTokenizer.from_pretrained(SD_ID, subfolder="tokenizer")
text_encoder = CLIPTextModel.from_pretrained(
    SD_ID, subfolder="text_encoder", torch_dtype=torch.float16
).to(DEVICE).eval()

prompts = ["image of " + id_to_caption[i] for i in range(40)] + [""]
EMPTY_PROMPT_ID = 40

with torch.no_grad():
    token_ids = tokenizer(
        prompts,
        max_length=tokenizer.model_max_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    ).input_ids.to(DEVICE)
    TEXT_EMBEDDINGS = text_encoder(token_ids, return_dict=False)[0].half()

del text_encoder
torch.cuda.empty_cache()

# Image preprocessing and precomputed VAE latents stored on CPU RAM
image_transform = transforms.Compose([
    transforms.Resize(RES, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RES),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
])

needed_image_ids = sorted(set(IMAGE_IDS[np.concatenate([train_rows, val_rows])].tolist()))
image_to_latent_row = {image_id: idx for idx, image_id in enumerate(needed_image_ids)}

vae = AutoencoderKL.from_pretrained(
    SD_ID, subfolder="vae", torch_dtype=torch.float16
).to(DEVICE).eval()
LATENT_SCALE = vae.config.scaling_factor

latent_param_chunks = []
with torch.no_grad():
    for start in tqdm(range(0, len(needed_image_ids), LATENT_BATCH_SIZE), desc="Encoding images with VAE to CPU"):
        batch_image_ids = needed_image_ids[start:start + LATENT_BATCH_SIZE]
        images = []
        for image_id in batch_image_ids:
            with Image.open(IMAGE_PATHS[int(image_id)]) as img:
                images.append(image_transform(img.convert("RGB")))
        pixel_batch = torch.stack(images).to(DEVICE, dtype=torch.float16)
        posterior_params = vae.encode(pixel_batch).latent_dist.parameters
        latent_param_chunks.append(posterior_params.cpu())

LATENT_PARAMS = torch.cat(latent_param_chunks, dim=0).contiguous()  # Stored on CPU RAM
del vae, latent_param_chunks
torch.cuda.empty_cache()

ROW_TO_LATENT = np.asarray(
    [image_to_latent_row.get(int(image_id), -1) for image_id in IMAGE_IDS],
    dtype=np.int64,
)
print("Cached VAE Latents on CPU RAM with shape:", tuple(LATENT_PARAMS.shape))
print("Stable Diffusion Backbone:", SD_ID)


## 6. EEG-Conditioned ControlNet Architecture
- `SubjectLayers`: Learns a subject-specific projection matrix for each of the 6 subjects to align individual skull/electrode differences.
- `EEGConditionEmbedding`: 1D convolutional layers transforming temporal EEG signals into 2D spatial feature maps (64x64).
- **Frozen UNet:** Keep Stable Diffusion UNet frozen and train only the ControlNet branches.
- `compute_loss`: Adds noise to target latents, predicts noise via ControlNet + UNet, and computes MSE loss.


In [ ]:
class SubjectLayers(nn.Module):
    def __init__(self, channels, output_dim, n_subjects):
        super().__init__()
        self.weights = nn.Parameter(torch.randn(n_subjects, channels, output_dim) / (channels ** 0.5))

    def forward(self, x, subject_indices):
        # x: [B, channels, time]
        _, channels, _ = x.shape
        weights = self.weights.gather(
            0,
            subject_indices.view(-1, 1, 1).expand(-1, channels, self.weights.shape[-1]),
        )
        return torch.einsum("bct,bcd->bdt", x, weights)

class EEGConditionEmbedding(nn.Module):
    def __init__(
        self,
        out_channels=320,
        channels=128,
        channel_sizes=(320, 640, 1280, 2560),
        strides=(5, 2, 2, 2),
        n_subjects=1,
    ):
        super().__init__()
        self.out_channels = out_channels
        self.subject_indices = None
        self.subj_layers = SubjectLayers(channels, channels, n_subjects)
        self.conv_in = nn.Conv1d(channels, channel_sizes[0], kernel_size=3, stride=1)
        self.blocks = nn.ModuleList()
        for i in range(len(channel_sizes) - 1):
            self.blocks.append(nn.Conv1d(channel_sizes[i], channel_sizes[i], kernel_size=3, padding=1))
            self.blocks.append(nn.Conv1d(
                channel_sizes[i],
                channel_sizes[i + 1],
                kernel_size=strides[i + 1] + 1,
                padding=1,
                stride=strides[i + 1],
            ))

    def forward(self, condition):
        x = condition.permute(0, 2, 1) if condition.shape[1] > condition.shape[2] else condition
        if self.subject_indices is None:
            raise RuntimeError("subject indices were not set before ControlNet forward.")
        if x.shape[-1] < 512:
            x = F.pad(x, (0, 512 - x.shape[-1]))
        elif x.shape[-1] > 512:
            x = x[..., :512]

        x = F.silu(self.conv_in(self.subj_layers(x, self.subject_indices)))
        for block in self.blocks:
            x = F.silu(block(x))

        if x.shape[1] % self.out_channels != 0:
            raise RuntimeError("Unexpected channel shape in EEG condition embedding.")
        x = x.reshape(
            x.shape[0],
            self.out_channels,
            x.shape[1] // self.out_channels,
            x.shape[2],
        ).permute(0, 1, 3, 2)

        x = F.pad(x, (0, max(0, 64 - x.shape[3]), 0, max(0, 64 - x.shape[2])))
        return x[:, :, :64, :64]

# Inspect existing checkpoints for smart resume / weight transfer
RESUME_CHECKPOINT = None
RESUME_SOURCE_PATH = None
SKIP_SUBJECT_WEIGHTS = False
N_SUBJECTS_MODEL = N_SUBJECTS

if os.path.exists(OUT_PATH):
    RESUME_SOURCE_PATH = OUT_PATH
elif os.path.exists(LEGACY_CHECKPOINT_PATH):
    RESUME_SOURCE_PATH = LEGACY_CHECKPOINT_PATH

if RESUME_SOURCE_PATH is not None:
    print("Found checkpoint; inspecting:", RESUME_SOURCE_PATH)
    RESUME_CHECKPOINT = torch.load(
        RESUME_SOURCE_PATH, map_location="cpu", weights_only=False
    )
    saved_control_state = RESUME_CHECKPOINT.get("controlnet", {})
    saved_subject_key = "controlnet_cond_embedding.subj_layers.weights"
    alternate_subject_key = "controlnet_cond_embedding.subject_layers.weights"
    if saved_subject_key not in saved_control_state and alternate_subject_key in saved_control_state:
        saved_subject_key = alternate_subject_key

    if saved_subject_key in saved_control_state:
        saved_n_subjects = int(saved_control_state[saved_subject_key].shape[0])
        saved_subject_values = RESUME_CHECKPOINT.get("subject_values")
        current_values = [int(v) for v in SUBJECT_VALUES]

        if saved_subject_values is not None:
            saved_subject_values = [int(v) for v in saved_subject_values]
            missing_subject_ids = sorted(set(current_values) - set(saved_subject_values))
            if missing_subject_ids:
                SKIP_SUBJECT_WEIGHTS = True
                print("Missing subject IDs in checkpoint; subject layer will be reinitialized.")
            else:
                SUBJECTS = np.asarray(
                    [saved_subject_values.index(int(v)) for v in SUBJECT_ORIGINAL],
                    dtype=np.int64,
                )
                N_SUBJECTS_MODEL = saved_n_subjects
        elif saved_n_subjects != N_SUBJECTS:
            SKIP_SUBJECT_WEIGHTS = True
            print(f"Subject count mismatch (Saved={saved_n_subjects}, Current={N_SUBJECTS}). Reinitializing subject layer.")
        else:
            N_SUBJECTS_MODEL = N_SUBJECTS

# Assemble UNet and ControlNet
unet = UNet2DConditionModel.from_pretrained(SD_ID, subfolder="unet")
controlnet = ControlNetModel.from_unet(unet)
controlnet.controlnet_cond_embedding = EEGConditionEmbedding(
    out_channels=controlnet.config.block_out_channels[0],
    n_subjects=N_SUBJECTS_MODEL,
)
unet.requires_grad_(False)
unet.to(DEVICE, dtype=torch.float16).eval()
unet.enable_gradient_checkpointing()

controlnet.to(DEVICE).train()
controlnet.enable_gradient_checkpointing()
noise_scheduler = DDPMScheduler.from_pretrained(SD_ID, subfolder="scheduler")

def compute_loss(rows, training=True):
    rows = np.asarray(rows, dtype=np.int64)
    eeg_batch = normalize_eeg(X[torch.as_tensor(rows)].to(DEVICE))
    subject_batch = torch.as_tensor(SUBJECTS[rows], dtype=torch.long, device=DEVICE)

    latent_rows = ROW_TO_LATENT[rows]
    if np.any(latent_rows < 0):
        raise RuntimeError("Missing image latent for row.")
    posterior_params = LATENT_PARAMS[torch.as_tensor(latent_rows)].to(DEVICE).float()
    mean, logvar = posterior_params.chunk(2, dim=1)
    logvar = logvar.clamp(-30.0, 20.0)
    latents = (mean + torch.exp(0.5 * logvar) * torch.randn_like(mean)) * LATENT_SCALE

    # Text dropout with PROP_EMPTY probability for Classifier-Free Guidance (CFG)
    prompt_ids = [
        EMPTY_PROMPT_ID
        if (training and random.random() < PROP_EMPTY)
        else int(CAPTIONS[row])
        for row in rows
    ]
    text_batch = TEXT_EMBEDDINGS[torch.as_tensor(prompt_ids, dtype=torch.long, device=DEVICE)]

    noise = torch.randn_like(latents)
    timesteps = torch.randint(
        0, noise_scheduler.config.num_train_timesteps,
        (len(rows),), device=DEVICE, dtype=torch.long,
    )
    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

    controlnet.controlnet_cond_embedding.subject_indices = subject_batch
    with torch.autocast(device_type="cuda", dtype=torch.float16):
        down_residuals, mid_residual = controlnet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=text_batch,
            controlnet_cond=eeg_batch,
            return_dict=False,
        )
        predicted_noise = unet(
            noisy_latents.half(),
            timesteps,
            encoder_hidden_states=text_batch,
            down_block_additional_residuals=[residual.half() for residual in down_residuals],
            mid_block_additional_residual=mid_residual.half(),
            return_dict=False,
        )[0]

    return F.mse_loss(predicted_noise.float(), noise.float())

print("ControlNet and EEG conditioning pipeline initialized.")


## 7. Training & Checkpointing Loop
- Optimization via **AdamW** with learning rate `1e-5`.
- **Mixed Precision (fp16)** via `GradScaler` with `Gradient Accumulation = 2`.
- Validation evaluation after each epoch, saving the best model to `best_gwit_model_6subjects.pt`.


In [ ]:
trainable_parameters = [p for p in controlnet.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=LR,
    betas=(0.9, 0.999),
    weight_decay=WEIGHT_DECAY,
    eps=1e-8,
)
scaler = torch.amp.GradScaler("cuda", enabled=True)

val_rng = np.random.RandomState(SEED + 1)
val_subset = val_rng.choice(val_rows, min(256, len(val_rows)), replace=False)
test_subset = np.random.RandomState(SEED + 2).choice(
    test_rows, min(N_STORE_TEST, len(test_rows)), replace=False
)

def make_checkpoint(epoch, global_step, elapsed_hours, val_loss_value=None):
    return {
        "controlnet": {k: v.detach().cpu().half() for k, v in controlnet.state_dict().items()},
        "optimizer": optimizer.state_dict(),
        "scaler": scaler.state_dict(),
        "epoch": int(epoch),
        "global_step": int(global_step),
        "elapsed_hours": float(elapsed_hours),
        "val_loss": None if val_loss_value is None else float(val_loss_value),
        "n_subjects": N_SUBJECTS_MODEL,
        "subject_values": SUBJECT_VALUES,
        "sd_id": SD_ID,
        "res": RES,
        "text_emb": TEXT_EMBEDDINGS.detach().cpu(),
        "id_to_caption": id_to_caption,
        "decoder": {
            "featnet": {k: v.detach().cpu() for k, v in featnet.state_dict().items()},
            "knn_pickle": pickle.dumps(knn),
        },
        "test": {
            "eeg": X[torch.as_tensor(test_subset)].clone(),
            "subject": SUBJECTS[test_subset].copy(),
            "image_path": [IMAGE_PATHS[int(IMAGE_IDS[row])] for row in test_subset],
            "class_name": [folder_label_map.get(IMAGE_WORDS[row], IMAGE_WORDS[row]) for row in test_subset],
        },
        "hparams": {
            "LR": LR, "EPOCHS": EPOCHS, "PROP_EMPTY": PROP_EMPTY,
            "BATCH_SIZE": BATCH_SIZE, "GRAD_ACCUM": GRAD_ACCUM,
            "split_sizes": (len(train_rows), len(val_rows), len(test_rows)),
        },
    }

@torch.no_grad()
def evaluate_validation_loss():
    controlnet.eval()
    losses = []
    cpu_rng_state = torch.random.get_rng_state()
    cuda_rng_state = torch.cuda.get_rng_state_all()
    random_state = random.getstate()
    np_state = np.random.get_state()
    torch.manual_seed(SEED + 123)
    torch.cuda.manual_seed_all(SEED + 123)
    random.seed(SEED + 123)
    np.random.seed(SEED + 123)
    for start in range(0, len(val_subset), BATCH_SIZE):
        rows = val_subset[start:start + BATCH_SIZE]
        if len(rows):
            losses.append(compute_loss(rows, training=False).item())
    torch.random.set_rng_state(cpu_rng_state)
    torch.cuda.set_rng_state_all(cuda_rng_state)
    random.setstate(random_state)
    np.random.set_state(np_state)
    controlnet.train()
    return float(np.mean(losses)) if losses else float("inf")

start_epoch = 1
global_step = 0
if RESUME_SOURCE_PATH is not None:
    print("Loading weights from:", RESUME_SOURCE_PATH)
    checkpoint = RESUME_CHECKPOINT or torch.load(RESUME_SOURCE_PATH, map_location="cpu", weights_only=False)
    saved_state = {}
    for key, value in checkpoint["controlnet"].items():
        normalized_key = key.replace(
            "controlnet_cond_embedding.subject_layers.",
            "controlnet_cond_embedding.subj_layers."
        )
        saved_state[normalized_key] = value

    current_state = controlnet.state_dict()
    subject_key = "controlnet_cond_embedding.subj_layers.weights"
    compatible_state = {}
    for key, saved_tensor in saved_state.items():
        if key in current_state and tuple(saved_tensor.shape) == tuple(current_state[key].shape):
            if key == subject_key and SKIP_SUBJECT_WEIGHTS:
                continue
            compatible_state[key] = saved_tensor

    controlnet.load_state_dict(compatible_state, strict=False)
    if not SKIP_SUBJECT_WEIGHTS and "optimizer" in checkpoint:
        try:
            optimizer.load_state_dict(checkpoint["optimizer"])
            start_epoch = int(checkpoint.get("epoch", 0)) + 1
            global_step = int(checkpoint.get("global_step", 0))
            print("Restored optimizer state. Resuming from epoch:", start_epoch)
        except Exception:
            print("Using fresh optimizer state.")
    else:
        print("Compatible non-subject weights loaded successfully.")
    del checkpoint

best_val_loss = float("inf")
if os.path.exists(BEST_PATH):
    best_cp = torch.load(BEST_PATH, map_location="cpu", weights_only=False)
    best_val_loss = float(best_cp.get("val_loss", float("inf")))
    del best_cp

run_start_time = time.time()
stop_training = False

for epoch in range(start_epoch, EPOCHS + 1):
    controlnet.train()
    shuffled_rows = np.random.permutation(train_rows)
    usable_count = (len(shuffled_rows) // BATCH_SIZE) * BATCH_SIZE
    shuffled_rows = shuffled_rows[:usable_count]
    epoch_losses = []
    optimizer.zero_grad(set_to_none=True)
    pending_micro_batches = 0

    progress = tqdm(
        range(0, len(shuffled_rows), BATCH_SIZE),
        desc=f"Epoch {epoch:02d}/{EPOCHS:02d}",
    )

    for batch_start in progress:
        rows = shuffled_rows[batch_start:batch_start + BATCH_SIZE]
        loss = compute_loss(rows, training=True)
        scaler.scale(loss / GRAD_ACCUM).backward()
        epoch_losses.append(float(loss.item()))
        pending_micro_batches += 1

        is_accumulation_boundary = (pending_micro_batches == GRAD_ACCUM)
        is_last_batch = (batch_start + BATCH_SIZE) >= len(shuffled_rows)

        if is_accumulation_boundary or is_last_batch:
            if pending_micro_batches < GRAD_ACCUM:
                scaler.unscale_(optimizer)
                correction = GRAD_ACCUM / pending_micro_batches
                for parameter in trainable_parameters:
                    if parameter.grad is not None:
                        parameter.grad.mul_(correction)
            else:
                scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(trainable_parameters, MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_micro_batches = 0
            global_step += 1

            if global_step % SAVE_EVERY_STEPS == 0:
                cp = make_checkpoint(epoch, global_step, (time.time() - run_start_time) / 3600)
                torch.save(cp, OUT_PATH)
                del cp

        progress.set_postfix(train_loss=f"{np.mean(epoch_losses[-20:]):.4f}", step=global_step)

        if (time.time() - run_start_time) >= TIME_BUDGET_H * 3600:
            stop_training = True
            break

    current_val_loss = evaluate_validation_loss()
    mean_train_loss = float(np.mean(epoch_losses)) if epoch_losses else float("nan")
    elapsed_hours = (time.time() - run_start_time) / 3600
    print(
        f"Epoch {epoch:02d} | train loss={mean_train_loss:.5f} | "
        f"val loss={current_val_loss:.5f} | elapsed={elapsed_hours:.2f} h"
    )

    cp = make_checkpoint(epoch, global_step, elapsed_hours, current_val_loss)
    torch.save(cp, OUT_PATH)

    if current_val_loss < best_val_loss:
        best_val_loss = current_val_loss
        torch.save(cp, BEST_PATH)
        print(f"Saved BEST model checkpoint: val_loss={best_val_loss:.5f}")

    del cp
    torch.cuda.empty_cache()

    if stop_training:
        print("Time budget reached. Checkpoint saved safely.")
        break

print("Training finished.")
print("Latest checkpoint:", OUT_PATH)
print("Best checkpoint:", BEST_PATH)


## 8. Evaluation & Verification Diagnostics
Print quantitative metrics (decoder accuracy, dataset split sizes, image overlap verification) and load the best saved checkpoint.


In [ ]:
print("Decoder validation class accuracy:", f"{decoder_accuracy(val_rows):.4f}")
print("Split sizes:", {
    "train": len(train_rows),
    "validation": len(val_rows),
    "test": len(test_rows),
})
print("Unique image overlap (Zero expected):", {
    "train-validation": image_overlap(train_rows, val_rows),
    "train-test": image_overlap(train_rows, test_rows),
    "validation-test": image_overlap(val_rows, test_rows),
})
print("Latest checkpoint exists:", os.path.isfile(OUT_PATH))
print("Best checkpoint exists:", os.path.isfile(BEST_PATH))

# Load best checkpoint weights
checkpoint_to_load = BEST_PATH if os.path.exists(BEST_PATH) else (OUT_PATH if os.path.exists(OUT_PATH) else None)
if checkpoint_to_load is not None:
    checkpoint = torch.load(checkpoint_to_load, map_location="cpu", weights_only=False)
    controlnet.load_state_dict(checkpoint["controlnet"], strict=True)
    controlnet.eval()
    print("Loaded ControlNet checkpoint successfully from:", os.path.basename(checkpoint_to_load))
    print("  - Best Epoch:", checkpoint.get("epoch"))
    print("  - Best Validation Loss:", checkpoint.get("val_loss"))


## 9. Visual Reconstruction & Inference Pipeline
**End-to-End Demonstration:**
1. Select an unseen EEG sample from the test set.
2. Predict the semantic prompt via k-NN and condition ControlNet with the EEG signal.
3. Run reverse diffusion sampling over 30 steps with Classifier-Free Guidance (CFG).
4. Decode latents with VAE and display a 3-panel comparison:
   **(Raw EEG Waveform + Ground Truth Image + AI Reconstructed Image from Brain Signals)**.


In [ ]:
@torch.no_grad()
def reconstruct_eeg_to_image(
    test_idx,
    num_inference_steps=30,
    guidance_scale=7.5,
    seed=42,
    save_fig_path=None
):
    """Reconstruct image from EEG trial and plot side-by-side comparison."""
    if seed is not None:
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    row = int(test_idx)
    raw_eeg = X[row].clone()
    subject_id = SUBJECTS[row]
    gt_image_path = IMAGE_PATHS[int(IMAGE_IDS[row])]
    true_label = folder_label_map.get(IMAGE_WORDS[row], "Unknown")
    pred_label_id = int(CAPTIONS[row])
    pred_label = id_to_caption.get(pred_label_id, "Unknown")

    # 1. Prepare conditioning
    eeg_batch = normalize_eeg(raw_eeg.unsqueeze(0)).to(DEVICE)
    subject_batch = torch.as_tensor([subject_id], dtype=torch.long, device=DEVICE)
    controlnet.controlnet_cond_embedding.subject_indices = subject_batch

    cond_text = TEXT_EMBEDDINGS[pred_label_id].unsqueeze(0).to(DEVICE)
    uncond_text = TEXT_EMBEDDINGS[EMPTY_PROMPT_ID].unsqueeze(0).to(DEVICE)

    # 2. Setup noise scheduler
    scheduler = copy.deepcopy(noise_scheduler)
    scheduler.set_timesteps(num_inference_steps, device=DEVICE)

    # 3. Initial noise latents
    latents = torch.randn((1, 4, RES // 8, RES // 8), device=DEVICE, dtype=torch.float32)
    latents = latents * scheduler.init_noise_sigma

    controlnet.eval()
    unet.eval()

    # 4. Reverse Diffusion Sampling Loop
    for t in tqdm(scheduler.timesteps, desc=f"Reconstructing #{row}", leave=False):
        latent_model_input = scheduler.scale_model_input(latents, t)

        with torch.autocast(device_type="cuda", dtype=torch.float16):
            down_residuals, mid_residual = controlnet(
                latent_model_input,
                t,
                encoder_hidden_states=cond_text,
                controlnet_cond=eeg_batch,
                return_dict=False
            )

            noise_pred_cond = unet(
                latent_model_input.half(),
                t,
                encoder_hidden_states=cond_text,
                down_block_additional_residuals=[r.half() for r in down_residuals],
                mid_block_additional_residual=mid_residual.half(),
                return_dict=False
            )[0]

            noise_pred_uncond = unet(
                latent_model_input.half(),
                t,
                encoder_hidden_states=uncond_text,
                down_block_additional_residuals=[r.half() for r in down_residuals],
                mid_block_additional_residual=mid_residual.half(),
                return_dict=False
            )[0]

            # Apply Classifier-Free Guidance (CFG)
            noise_pred = noise_pred_uncond + guidance_scale * (noise_pred_cond - noise_pred_uncond)

        latents = scheduler.step(noise_pred.float(), t, latents).prev_sample

    # 5. Decode latents to RGB image via VAE
    vae = AutoencoderKL.from_pretrained(SD_ID, subfolder="vae", torch_dtype=torch.float16).to(DEVICE).eval()
    with torch.no_grad():
        decoded_image = vae.decode(latents.half() / LATENT_SCALE).sample
        decoded_image = (decoded_image / 2.0 + 0.5).clamp(0.0, 1.0)
        decoded_image = decoded_image.cpu().permute(0, 2, 3, 1).float().numpy()[0]
        reconstructed_pil = Image.fromarray((decoded_image * 255).astype(np.uint8))

    del vae
    torch.cuda.empty_cache()

    # 6. Plot 3-panel comparison figure
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), dpi=130)

    # (a) Raw EEG Waveform
    time_axis = np.arange(raw_eeg.shape[1])
    for ch in [0, 16, 32, 64, 96, 127]:
        axes[0].plot(time_axis, raw_eeg[ch].cpu().numpy(), alpha=0.7, label=f"Ch {ch+1}")
    axes[0].set_title(f"Visual Evoked EEG\n(Subject: {SUBJECT_ORIGINAL[row]})", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Time Steps (440)", fontsize=9)
    axes[0].set_ylabel("Amplitude", fontsize=9)
    axes[0].grid(True, linestyle="--", alpha=0.4)
    axes[0].legend(loc="upper right", fontsize=7, ncol=2)

    # (b) Ground Truth Image
    if gt_image_path and os.path.isfile(gt_image_path):
        axes[1].imshow(Image.open(gt_image_path).convert("RGB"))
    else:
        axes[1].text(0.5, 0.5, "Image not found", ha="center")
    axes[1].set_title(f"Ground Truth Image\nClass: {true_label}", fontsize=11, fontweight="bold")
    axes[1].axis("off")

    # (c) Reconstructed Image from Brain
    axes[2].imshow(reconstructed_pil)
    match_tag = "✓ Match" if true_label == pred_label else "≈ Semantic"
    axes[2].set_title(f"Reconstructed from Brain\nDecoded: {pred_label} ({match_tag})", fontsize=11, fontweight="bold")
    axes[2].axis("off")

    plt.tight_layout()
    if save_fig_path:
        plt.savefig(save_fig_path, bbox_inches="tight")
    plt.show()

    return reconstructed_pil

# Run reconstruction on the first test sample
if len(test_subset) > 0:
    print(f"Generating reconstruction test for sample index {test_subset[0]}...")
    try:
        sample_img = reconstruct_eeg_to_image(
            test_idx=test_subset[0],
            num_inference_steps=30,
            guidance_scale=7.5,
            seed=42
        )
    except Exception as exc:
        print(f"Inference can run after training produces checkpoint: {exc}")
else:
    print("Ready for inference once test data is available.")
